# Broadcasting 广播机制

Broadcasting 称为广播机制(或自动扩展机制)，它是一种轻量级的张量复制手段，在逻
辑上扩展张量数据的形状，但是只会在需要时才会执行实际存储复制操作。对于大部分场
景，Broadcasting 机制都能通过优化手段避免实际复制数据而完成逻辑运算，从而相对于
tf.tile 函数，减少了大量计算代价。
对于所有长度为 1 的维度，Broadcasting 的效果和 tf.tile 一样，都能在此维度上逻辑复
制数据若干份，区别在于 *tf.tile 会创建一个新的张量，执行复制 IO 操作*，并保存复制后的
张量数据，而 Broadcasting 并不会立即复制数据，它会在逻辑上改变张量的形状，使得视
图上变成了复制后的形状。Broadcasting 会通过深度学习框架的优化手段避免实际复制数据
而完成逻辑运算，至于怎么实现的用户不必关心，对于用户来说，Broadcasting 和 tf.tile 复
制的最终效果是一样的，操作对用户透明，但是 Broadcasting 机制节省了大量计算资源，
建议在运算过程中尽可能地利用 Broadcasting 机制提高计算效率。

In [2]:
import tensorflow as tf

# 直接将 shape 为［2,3］与［3］的b相加也是合法的，
x = tf.random.normal([2, 4])
w = tf.random.normal([4, 3])
b = tf.zeros([3])

y = x @ w + b
y

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[-1.5073059, 11.632852 , -2.456657 ],
       [-1.7758336, -2.235626 , -0.5183656]], dtype=float32)>

上述加法并没有发生逻辑错误，那么它是怎么实现的呢？这是因为它自动调用 Broadcasting
函数 tf.broadcast_to(x, new_shape)，将两者 shape 扩张为相同的[2,3]，即上式可以等效为:

In [3]:
y = x @ w + tf.broadcast_to(b, [2, 3])  # 手动扩展并相加
y

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[-1.5073059, 11.632852 , -2.456657 ],
       [-1.7758336, -2.235626 , -0.5183656]], dtype=float32)>

Broadcasting 机制的核心思想是普适性，即同一份数据能普遍适合于其他位置。在验证
普适性之前，需要先将张量 shape 靠右对齐，然后进行普适性判断：对于长度为 1 的维
度，默认这个数据普遍适合于当前维度的其他位置；对于不存在的维度，则在增加新维度
后默认当前数据也是普适于新维度的，从而可以扩展为更多维度数、任意长度的张量形
状。

In [ ]:
# 通过 tf.broadcast_to(x, new_shape)函数可以显式地执行自动扩展功能，将现有 shape 扩张为 new_shape，实现如下
A = tf.random.normal([32, 1])
new_A = tf.broadcast_to(A, [2, 32, 32, 3])  # 扩展为4D张量
new_A.shape  # 在普适性原则的指导下，Broadcasting 机制变得直观、好理解


TensorShape([2, 32, 32, 3])

简单测试一下基本运算符的自动 Broadcasting 机制

In [ ]:
a = tf.random.normal([2, 32, 32, 1])
b = tf.random.normal([32, 32])
a + b, a - b, a * b, a / b  # 测试加减乘除运算的 Broadcasting 机制

(<tf.Tensor: shape=(2, 32, 32, 32), dtype=float32, numpy=
 array([[[[-1.0921267 , -0.7526891 , -0.96326387, ..., -1.8696666 ,
            0.31756493, -0.23375197],
          [-0.01376149, -1.1194835 ,  0.5601871 , ...,  1.322588  ,
           -1.294884  , -0.7247921 ],
          [ 1.2078228 ,  1.097604  ,  1.514828  , ..., -0.40657553,
            0.26529506, -1.3028965 ],
          ...,
          [ 0.00870293, -2.1364245 , -0.82287514, ..., -1.268199  ,
           -0.16116422, -1.4880497 ],
          [ 0.63143075,  0.2970758 , -0.2989279 , ...,  4.0005937 ,
            2.1572814 ,  1.5288526 ],
          [-1.8975649 , -5.2844906 , -2.7980137 , ..., -2.1371427 ,
           -5.1251287 , -3.5462372 ]],
 
         [[-0.9937638 , -0.6543262 , -0.86490095, ..., -1.7713037 ,
            0.41592786, -0.13538904],
          [-0.38228276, -1.4880047 ,  0.19166583, ...,  0.95406675,
           -1.6634052 , -1.0933133 ],
          [ 0.35198182,  0.241763  ,  0.658987  , ..., -1.2624166 ,
        